In [11]:
"""
============================================================
  Unemployment in India – Exploratory Data Analysis
  Author  : [Your Name]
  Dataset : Unemployment_in_India.csv (CMIE via Kaggle)
============================================================
"""

# ── 0. Imports ────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# ── 1. Aesthetics ─────────────────────────────────────────
PALETTE   = sns.color_palette("Set2")
COVID_CLR = "#e74c3c"
PRE_CLR   = "#3498db"
RURAL_CLR = "#27ae60"
URBAN_CLR = "#e67e22"

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": "#f9f9f9",
    "axes.facecolor":   "#f9f9f9",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})

# ── 2. Load & Clean ───────────────────────────────────────
df = pd.read_csv("Unemployment_in_India.csv")

# Strip column-name whitespace
df.columns = df.columns.str.strip()

# Parse dates (mixed order — try both formats)
df["Date"] = pd.to_datetime(df["Date"].str.strip(),
                            format="%d-%m-%Y", errors="coerce")
df.dropna(subset=["Date"], inplace=True)
df.sort_values("Date", inplace=True)

# Rename verbose columns
df.rename(columns={
    "Estimated Unemployment Rate (%)":       "Unemployment_Rate",
    "Estimated Employed":                    "Employed",
    "Estimated Labour Participation Rate (%)":"Labour_Part_Rate",
}, inplace=True)

# Drop rows where all numeric cols are NaN
df.dropna(subset=["Unemployment_Rate", "Employed", "Labour_Part_Rate"],
          how="all", inplace=True)

# Clean Area column (keep only Rural / Urban)
df["Area"] = df["Area"].str.strip()
df = df[df["Area"].isin(["Rural", "Urban"])]

# Covid flag  (lockdown announced 25-Mar-2020)
COVID_START = pd.Timestamp("2020-03-25")
df["Period"] = np.where(df["Date"] >= COVID_START, "COVID-19", "Pre-COVID")

print("=" * 55)
print("  DATASET SUMMARY")
print("=" * 55)
print(f"  Rows       : {len(df)}")
print(f"  Date range : {df['Date'].min().date()}  →  {df['Date'].max().date()}")
print(f"  Regions    : {df['Region'].nunique()}")
print(f"  Areas      : {sorted(df['Area'].unique())}")
print()
print(df[["Unemployment_Rate","Employed","Labour_Part_Rate"]].describe().round(2))

# ── 3. Monthly national average ───────────────────────────
monthly = (df.groupby("Date")
             .agg(Avg_UR=("Unemployment_Rate","mean"),
                  Avg_LPR=("Labour_Part_Rate","mean"))
             .reset_index())

# ── 4. Pre / Post COVID averages ─────────────────────────
covid_summary = (df.groupby("Period")
                   .agg(Mean_UR =("Unemployment_Rate","mean"),
                        Median_UR=("Unemployment_Rate","median"),
                        Mean_LPR =("Labour_Part_Rate","mean"))
                   .round(2))
print("\n  PRE vs COVID average unemployment")
print(covid_summary)

# ── 5. Region-level averages ─────────────────────────────
region_avg = (df.groupby("Region")["Unemployment_Rate"]
                .mean()
                .sort_values(ascending=False)
                .reset_index())

# ── 6. Rural vs Urban over time ───────────────────────────
rural_urban = (df.groupby(["Date","Area"])["Unemployment_Rate"]
                 .mean()
                 .unstack("Area")
                 .reset_index())

# ══════════════════════════════════════════════════════════
#  FIGURE 1 – Overview dashboard (2 × 2)
# ══════════════════════════════════════════════════════════
fig1, axes = plt.subplots(2, 2, figsize=(16, 10))
fig1.suptitle("Unemployment in India – Overview", fontsize=18, fontweight="bold", y=1.01)

# (A) National trend with COVID band
ax = axes[0, 0]
ax.fill_between(monthly["Date"], monthly["Avg_UR"],
                alpha=0.15, color=PRE_CLR)
ax.plot(monthly["Date"], monthly["Avg_UR"],
        color=PRE_CLR, linewidth=2.2, label="Avg Unemployment Rate")
ax.axvline(COVID_START, color=COVID_CLR, linestyle="--", linewidth=1.8,
           label="COVID Lockdown (Mar 2020)")
ax.axvspan(COVID_START, monthly["Date"].max(),
           alpha=0.08, color=COVID_CLR)
ax.set_title("A. National Monthly Unemployment Rate (%)")
ax.set_ylabel("Unemployment Rate (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
ax.legend(fontsize=9)

# (B) Pre vs COVID box plot
ax = axes[0, 1]
sns.boxplot(data=df, x="Period", y="Unemployment_Rate",
            palette={"Pre-COVID": PRE_CLR, "COVID-19": COVID_CLR},
            ax=ax, width=0.5, flierprops=dict(marker="o", alpha=0.4))
ax.set_title("B. Unemployment: Pre-COVID vs During COVID")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("")
for p, val in zip(ax.patches,
                  [covid_summary.loc[p, "Mean_UR"]
                   for p in covid_summary.index]):
    pass                                          # means annotated separately

# Annotate means
for i, period in enumerate(["Pre-COVID", "COVID-19"]):
    mean_val = covid_summary.loc[period, "Mean_UR"]
    ax.text(i, mean_val + 1, f"μ = {mean_val:.1f}%",
            ha="center", fontsize=10, color="black", fontweight="bold")

# (C) Top / Bottom 10 states
ax = axes[1, 0]
top5    = region_avg.head(10)
bottom5 = region_avg.tail(10)
combo   = pd.concat([top5, bottom5]).drop_duplicates()
colors  = [COVID_CLR if x in top5["Region"].values else PRE_CLR
           for x in combo["Region"]]
bars = ax.barh(combo["Region"], combo["Unemployment_Rate"],
               color=colors, edgecolor="white")
ax.set_title("C. Average Unemployment Rate by State (Top & Bottom 10)")
ax.set_xlabel("Avg Unemployment Rate (%)")
legend_elements = [Patch(facecolor=COVID_CLR, label="Top 10 (Highest)"),
                   Patch(facecolor=PRE_CLR,   label="Bottom 10 (Lowest)")]
ax.legend(handles=legend_elements, fontsize=9)
ax.invert_yaxis()

# (D) Labour Participation Rate trend
ax = axes[1, 1]
ax.plot(monthly["Date"], monthly["Avg_LPR"],
        color="#8e44ad", linewidth=2.2)
ax.fill_between(monthly["Date"], monthly["Avg_LPR"],
                alpha=0.12, color="#8e44ad")
ax.axvline(COVID_START, color=COVID_CLR, linestyle="--", linewidth=1.8)
ax.set_title("D. Labour Participation Rate Over Time (%)")
ax.set_ylabel("Labour Participation Rate (%)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))

for ax_ in axes.flat:
    ax_.tick_params(axis="x", rotation=30)

fig1.tight_layout()
fig1.savefig("fig1_overview.png", dpi=150, bbox_inches="tight")
print("\n  ✔  fig1_overview.png saved")

# ══════════════════════════════════════════════════════════
#  FIGURE 2 – Rural vs Urban deep-dive
# ══════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
fig2.suptitle("Rural vs Urban Unemployment Analysis", fontsize=16,
              fontweight="bold")

# (A) Time-series comparison
ax = axes2[0]
for area, color, ls in [("Rural", RURAL_CLR, "-"),
                         ("Urban", URBAN_CLR, "--")]:
    if area in rural_urban.columns:
        ax.plot(rural_urban["Date"], rural_urban[area],
                color=color, linewidth=2.2, linestyle=ls, label=area)
ax.axvline(COVID_START, color=COVID_CLR, linestyle=":", linewidth=1.8,
           label="COVID Lockdown")
ax.axvspan(COVID_START, rural_urban["Date"].max(),
           alpha=0.07, color=COVID_CLR)
ax.set_title("A. Rural vs Urban – Monthly Trend")
ax.set_ylabel("Avg Unemployment Rate (%)")
ax.legend()
ax.tick_params(axis="x", rotation=30)

# (B) Violin plot split by Period
ax = axes2[1]
df_rv = df[df["Area"].isin(["Rural","Urban"])]
sns.violinplot(data=df_rv, x="Area", y="Unemployment_Rate",
               hue="Period",
               palette={"Pre-COVID": PRE_CLR, "COVID-19": COVID_CLR},
               split=True, inner="quartile", ax=ax)
ax.set_title("B. Unemployment Distribution: Rural vs Urban")
ax.set_ylabel("Unemployment Rate (%)")
ax.set_xlabel("")

fig2.tight_layout()
fig2.savefig("fig2_rural_urban.png", dpi=150, bbox_inches="tight")
print("  ✔  fig2_rural_urban.png saved")

# ══════════════════════════════════════════════════════════
#  FIGURE 3 – COVID shock heatmap (month × state)
# ══════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(18, 10))

df["YearMonth"] = df["Date"].dt.to_period("M")
heatmap_data = (df.groupby(["Region","YearMonth"])["Unemployment_Rate"]
                  .mean()
                  .unstack("YearMonth"))

# Sort states by overall mean
heatmap_data = heatmap_data.loc[heatmap_data.mean(axis=1)
                                             .sort_values(ascending=False).index]

sns.heatmap(heatmap_data, cmap="YlOrRd", ax=ax3,
            cbar_kws={"label": "Unemployment Rate (%)"},
            linewidths=0.3, linecolor="#eeeeee")

# Mark COVID start column
cols = [str(c) for c in heatmap_data.columns]
covid_col = next((i for i, c in enumerate(cols) if "2020-03" <= c), None)
if covid_col is not None:
    ax3.axvline(covid_col, color=COVID_CLR, linewidth=2.5,
                label="COVID start")
    ax3.text(covid_col + 0.3, -0.7, "◀ COVID", color=COVID_CLR,
             fontsize=10, fontweight="bold", transform=ax3.get_xaxis_transform())

ax3.set_title("State-wise Monthly Unemployment Heatmap", fontsize=15,
              fontweight="bold", pad=14)
ax3.set_xlabel("Month")
ax3.set_ylabel("State / Region")
ax3.tick_params(axis="x", rotation=70, labelsize=8)
ax3.tick_params(axis="y", labelsize=9)

fig3.tight_layout()
fig3.savefig("fig3_heatmap.png", dpi=150, bbox_inches="tight")
print("  ✔  fig3_heatmap.png saved")

# ══════════════════════════════════════════════════════════
#  FIGURE 4 – Key insights summary card
# ══════════════════════════════════════════════════════════
# Compute stats for the card
pre   = df[df["Period"] == "Pre-COVID"]["Unemployment_Rate"]
post  = df[df["Period"] == "COVID-19"]["Unemployment_Rate"]
peak_row = df.loc[df["Unemployment_Rate"].idxmax()]
hardest_state = region_avg.iloc[0]

fig4, ax4 = plt.subplots(figsize=(12, 7))
ax4.axis("off")
fig4.patch.set_facecolor("#1a1a2e")

insights = [
    ("📊 Dataset",
     f"{len(df):,} records across {df['Region'].nunique()} states  |  "
     f"{df['Date'].min().strftime('%b %Y')} – {df['Date'].max().strftime('%b %Y')}"),
    ("📈 Pre-COVID avg unemployment",
     f"{pre.mean():.1f}%  (median {pre.median():.1f}%)"),
    ("🦠 COVID-era avg unemployment",
     f"{post.mean():.1f}%  (median {post.median():.1f}%)  "
     f"→  +{post.mean()-pre.mean():.1f} pp spike"),
    ("📍 Peak unemployment",
     f"{peak_row['Unemployment_Rate']:.1f}%  in  {peak_row['Region']}  "
     f"({peak_row['Date'].strftime('%b %Y')})"),
    ("🏆 Hardest-hit state (overall avg)",
     f"{hardest_state['Region']}  —  {hardest_state['Unemployment_Rate']:.1f}%"),
    ("🌾 Rural vs Urban",
     f"Urban unemployment ({df[df['Area']=='Urban']['Unemployment_Rate'].mean():.1f}%) "
     f"typically higher than Rural "
     f"({df[df['Area']=='Rural']['Unemployment_Rate'].mean():.1f}%) "
     f"but gap widens sharply during COVID"),
    ("💡 Policy implication",
     "Urban informal sector & migrant labour most vulnerable;\n"
     "    targeted cash transfers + rural MGNREGS expansion recommended"),
]

y_pos = 0.95
ax4.text(0.5, y_pos, "KEY FINDINGS & POLICY INSIGHTS",
         transform=ax4.transAxes, fontsize=16, fontweight="bold",
         color="#f0f0f0", ha="center", va="top")
y_pos -= 0.10

for icon_label, detail in insights:
    ax4.text(0.04, y_pos, icon_label,
             transform=ax4.transAxes, fontsize=11, fontweight="bold",
             color="#f39c12", va="top")
    ax4.text(0.04, y_pos - 0.07, f"  {detail}",
             transform=ax4.transAxes, fontsize=10,
             color="#ecf0f1", va="top", wrap=True)
    y_pos -= 0.13

fig4.savefig("fig4_insights.png", dpi=150, bbox_inches="tight",
             facecolor=fig4.get_facecolor())
print("  ✔  fig4_insights.png saved")

# ──────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  ANALYSIS COMPLETE – 4 figures saved")
print("=" * 55)
plt.close("all")

  DATASET SUMMARY
  Rows       : 740
  Date range : 2019-05-31  →  2020-06-30
  Regions    : 28
  Areas      : ['Rural', 'Urban']

       Unemployment_Rate     Employed  Labour_Part_Rate
count             740.00       740.00            740.00
mean               11.79   7204460.03             42.63
std                10.72   8087988.43              8.11
min                 0.00     49420.00             13.33
25%                 4.66   1190404.50             38.06
50%                 8.35   4744178.50             41.16
75%                15.89  11275489.50             45.50
max                76.74  45777509.00             72.57

  PRE vs COVID average unemployment
           Mean_UR  Median_UR  Mean_LPR
Period                                 
COVID-19     17.77      14.52     39.33
Pre-COVID     9.51       7.12     43.89

  ✔  fig1_overview.png saved
  ✔  fig2_rural_urban.png saved
  ✔  fig3_heatmap.png saved
  ✔  fig4_insights.png saved

  ANALYSIS COMPLETE – 4 figures saved
